# Tutorial 6: Mixed Precision Quantization Search with Mase and Optuna

In this tutorial, we'll see how Mase can be integrated with Optuna, the popular hyperparameter optimization framework, to search for a Bert model optimized for sequence classification on the IMDb dataset. We'll take the Optuna-generated model and import it into Mase, then run the CompressionPipeline to prepare the model for edge deployment by quantizing and pruning its weights.

As we'll see, running Architecture Search with Mase/Optuna involves the following steps.

1. **Define the search space**: this is a dictionary containing the range of values for each parameter at each layer in the model.

2. **Write the model constructor**: this is a function which uses Optuna utilities to sample a model from the search space, and constructs the model using transformers from_config class method.

3. **Write the objective function**: this function calls on the model constructor defined in Step 2 and defines the training/evaluation setup for each search iteration.

4. **Go!** Choose an Optuna sampler, create a study and launch the search.

In [1]:
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

## Importing the model

If you are starting from scratch, you can load the Bert checkpoint directly from HuggingFace.

In [2]:
from transformers import AutoModel

model = AutoModel.from_pretrained(checkpoint)

If you have previously ran the tutorial on Neural Architecture Search (NAS), run the following cell to import the best model obtained from the search process.

In [ ]:
from pathlib import Path
import dill

with open(f"{Path.home()}/tutorial_5_best_model.pkl", "rb") as f:
    base_model = dill.load(f)

First, fetch the dataset using the `get_tokenized_dataset` utility.

In [ ]:
from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

## 1. Defining the Search Space

We'll start by defining a search space, i.e. enumerating the possible combinations of hyperparameters that Optuna can choose during search. We'll explore the following range of values for the model's hidden size, intermediate size, number of layers and number of heads.

In [5]:
import torch
from chop.nn.quantized.modules.linear import (
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    LinearBlockMinifloat,
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
    LinearBinaryResidualSign,
)

search_space = {
    "linear_layer_choices": [
        torch.nn.Linear,
        LinearInteger,
    ],
}

## 2. Writing a Model Constructor

We define the following function, which will get called in each iteration of the search process. The function is passed the `trial` argument, which is an Optuna object that comes with many functionalities - see the [Trial documentation](https://optuna.readthedocs.io/en/stable/reference/trial.html) for more details. Here, we use the `trial.suggest_categorical` function, which triggers the chosen sampler to choose a layer type. The suggested integer is the index into the search space for each parameter, which we defined in the previous cell.

In [6]:
from chop.tools.utils import deepsetattr
from copy import deepcopy


def construct_model(trial):

    # Fetch the model
    trial_model = deepcopy(base_model)

    # Quantize layers according to optuna suggestions
    for name, layer in trial_model.named_modules():
        if isinstance(layer, torch.nn.Linear):
            new_layer_cls = trial.suggest_categorical(
                f"{name}_type",
                search_space["linear_layer_choices"],
            )

            if new_layer_cls == torch.nn.Linear:
                continue

            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
            }

            # If the chosen layer is integer, define the low precision config
            if new_layer_cls == LinearInteger:
                kwargs["config"] = {
                    "data_in_width": 8,
                    "data_in_frac_width": 4,
                    "weight_width": 8,
                    "weight_frac_width": 4,
                    "bias_width": 8,
                    "bias_frac_width": 4,
                }
            # elif... (other precisions)

            # Create the new layer (copy the weights)
            new_layer = new_layer_cls(**kwargs)
            new_layer.weight.data = layer.weight.data

            # Replace the layer in the model
            deepsetattr(trial_model, name, new_layer)

    return trial_model

## 3. Defining the Objective Function

Next, we define the objective function for the search, which gets called on each trial. In each trial, we create a new model instace with chosen hyperparameters according to the defined sampler. We then use the `get_trainer` utility in Mase to run a training loop on the IMDb dataset for a number of epochs. Finally, we use `evaluate` to report back the classification accuracy on the test split.

In [7]:
from chop.tools import get_trainer
import random


def objective(trial):

    # Define the model
    model = construct_model(trial)

    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    trial.set_user_attr("model", model)

    return eval_results["eval_accuracy"]

## 4. Launching the Search

Optuna provides a number of samplers, for example:

* **GridSampler**: iterates through every possible combination of hyperparameters in the search space
* **RandomSampler**: chooses a random combination of hyperparameters in each iteration
* **TPESampler**: uses Tree-structured Parzen Estimator algorithm to choose hyperparameter values.

You can define the chosen sampler by simply importing from `optuna.samplers` as below.

In [8]:
from optuna.samplers import GridSampler, RandomSampler, TPESampler

sampler = RandomSampler()

With all the pieces in place, we can launch the search as follows. The number of trials is set to 1 so you can go get a coffee for 10 minutes, then proceed with the tutorial. However, this will essentially be a random model - for better results, set this to 100 and leave it running overnight!

In [ ]:
import optuna

study = optuna.create_study(
    direction="maximize",
    study_name="bert-tiny-nas-study",
    sampler=sampler,
)

study.optimize(
    objective,
    n_trials=1,
    timeout=60 * 60 * 24,
)

# Task 1

In [ ]:
search_space = {
    "linear_layer_choices": [torch.nn.Linear, LinearInteger],
    "linear_layer_data_width": [8, 16, 32],
    "linear_layer_frac_width": [2, 4, 8],
}

In [ ]:
def construct_model(trial):
    # get base model
    trial_model = deepcopy(base_model)

    # quantize layers according to optuna suggestions
    for name, layer in trial_model.named_modules():
        if isinstance(layer, torch.nn.Linear):
            # choose layer type
            new_layer_cls = trial.suggest_categorical(f"{name}_type", search_space["linear_layer_choices"])
            if new_layer_cls == torch.nn.Linear:
                continue
            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
            }

            # choose layer precision
            if new_layer_cls == LinearInteger:
                new_layer_data_width = trial.suggest_categorical(f"{name}_data_width", search_space["linear_layer_data_width"])
                new_layer_frac_width = trial.suggest_categorical(f"{name}_frac_width", search_space["linear_layer_frac_width"])

                kwargs["config"] = {
                    "data_in_width": new_layer_data_width,
                    "data_in_frac_width": new_layer_frac_width,
                    "weight_width": new_layer_data_width,
                    "weight_frac_width": new_layer_frac_width,
                    "bias_width": new_layer_data_width,
                    "bias_frac_width": new_layer_frac_width,
                }

            # replace layer
            new_layer = new_layer_cls(**kwargs)
            new_layer.weight.data = layer.weight.data
            deepsetattr(trial_model, name, new_layer)

    return trial_model

In [ ]:
data = []

def objective(trial):
    # model, train, evaluate
    model = construct_model(trial)
    trainer = get_trainer(model=model, tokenized_dataset=dataset, tokenizer=tokenizer, evaluate_metric="accuracy", num_train_epochs=1)
    trainer.train()
    eval_results = trainer.evaluate()

    # save model
    trial.set_user_attr("model", model)

    # report statistics
    data.append({"task": "task1", "trial": len(data), "acc": eval_results["eval_accuracy"]})

    return eval_results["eval_accuracy"]

In [ ]:
from pathlib import Path
from google.colab import files
import pandas as pd

# create and run study
sampler = TPESampler()
study = optuna.create_study(direction="maximize", study_name="bert-tiny-nas-study", sampler=sampler)
study.optimize(objective, n_trials=100, timeout=3600*24)

# save all trial information
df = pd.DataFrame(data)
df.to_pickle(f"{Path.home()}/tutorial6_task1.pickle")
files.download(f"{Path.home()}/tutorial6_task1.pickle")

# Task 2

In [ ]:
precision_dict = {
    "LinearMinifloatDenorm": LinearMinifloatDenorm,
    "LinearMinifloatIEEE": LinearMinifloatIEEE,
    "LinearLog": LinearLog,
    "LinearBlockFP": LinearBlockFP,
    # "LinearBlockMinifloat": LinearBlockMinifloat,
    "LinearBlockLog": LinearBlockLog,
    "LinearBinary": LinearBinary,
    "LinearBinaryScaling": LinearBinaryScaling,
    # "LinearBinaryResidualSign": LinearBinaryResidualSign,
}

In [ ]:
def construct_model(trial, search_space):
    # get base model
    trial_model = deepcopy(base_model)

    # quantize layers according to optuna suggestions
    for name, layer in trial_model.named_modules():
        if isinstance(layer, torch.nn.Linear):
            # choose layer type
            new_layer_cls = trial.suggest_categorical(f"{name}_type", search_space["linear_layer_choices"])
            if new_layer_cls == torch.nn.Linear:
                continue
            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
            }

            # choose layer precision
            if new_layer_cls == LinearInteger:
                data_width = [8, 16, 32]
                frac_width = [2, 4, 8]

                new_layer_data_width = data_width[trial.suggest_int(f"{name}_data_width", 0, len(data_width)-1)]
                new_layer_frac_width = frac_width[trial.suggest_int(f"{name}_frac_width", 0, len(frac_width)-1)]

                kwargs["config"] = {
                    "data_in_width": new_layer_data_width,
                    "data_in_frac_width": new_layer_frac_width,
                    "weight_width": new_layer_data_width,
                    "weight_frac_width": new_layer_frac_width,
                    "bias_width": new_layer_data_width,
                    "bias_frac_width": new_layer_frac_width,
                }

            elif new_layer_cls == LinearMinifloatIEEE:
                data_width = [8, 16, 32]
                exponent_width = [2, 4, 8]

                new_laye_data_width = data_width[trial.suggest_int(f"{name}_data_width", 0, len(data_width)-1)]
                new_layer_exponent_width = exponent_width[trial.suggest_int(f"{name}_exponent_width", 0, len(exponent_width)-1)]
                new_layer_bias = None

                kwargs["config"] = {
                    "data_in_width": new_laye_data_width,
                    "data_in_exponent_width": new_layer_exponent_width,
                    "data_in_exponent_bias": new_layer_bias,
                    "weight_width": new_laye_data_width,
                    "weight_exponent_width": new_layer_exponent_width,
                    "weight_exponent_bias": new_layer_bias,
                    "bias_width": new_laye_data_width,
                    "bias_exponent_width": new_layer_exponent_width,
                    "bias_exponent_bias": new_layer_bias,
                }

            elif new_layer_cls == LinearMinifloatDenorm:
                data_width = [8, 16, 32]
                exponent_width = [2, 4, 8]

                new_layer_data_width = data_width[trial.suggest_int(f"{name}_data_width", 0, len(data_width)-1)]
                new_layer_exponent_width = exponent_width[trial.suggest_int(f"{name}_exponent_width", 0, len(exponent_width)-1)]
                new_layer_bias = None

                kwargs["config"] = {
                    "data_in_width": new_layer_data_width,
                    "data_in_exponent_width": new_layer_exponent_width,
                    "data_in_exponent_bias": new_layer_bias,
                    "weight_width": new_layer_data_width,
                    "weight_exponent_width": new_layer_exponent_width,
                    "weight_exponent_bias": new_layer_bias,
                    "bias_width": new_layer_data_width,
                    "bias_exponent_width": new_layer_exponent_width,
                    "bias_exponent_bias": new_layer_bias,
                }

            elif new_layer_cls == LinearLog:
                data_width = [8, 16, 32]
                bias_width = [-1, 0, 1]

                new_layer_data_width = data_width[trial.suggest_int(f"{name}_data_width", 0, len(data_width)-1)]
                new_layer_bias_width = bias_width[trial.suggest_int(f"{name}_bias_width", 0, len(bias_width)-1)]

                kwargs["config"] = {
                    "data_in_width": new_layer_data_width,
                    "data_in_exponent_bias": new_layer_bias_width,
                    "weight_width": new_layer_data_width,
                    "weight_exponent_bias": new_layer_bias_width,
                    "bias_width": new_layer_data_width,
                    "bias_exponent_bias": new_layer_bias_width,
                }

            elif new_layer_cls == LinearBlockFP:
                data_width = [4, 8, 16]
                exponent_width = [4, 8, 16]
                block_size = [8, 16, 32]

                new_layer_data_width = data_width[trial.suggest_int(f"{name}_data_width", 0, len(data_width)-1)]
                new_layer_exponent_width = exponent_width[trial.suggest_int(f"{name}_exponent_width", 0, len(exponent_width)-1)]
                new_layer_block_size = block_size[trial.suggest_int(f"{name}_block_size", 0, len(block_size)-1)]
                new_layer_exponent_bias = None

                kwargs["config"] = {
                    "data_in_width": new_layer_data_width,
                    "data_in_exponent_width": new_layer_exponent_width,
                    "data_in_exponent_bias": new_layer_exponent_bias,
                    "data_in_block_size": new_layer_block_size,
                    "weight_width": new_layer_data_width,
                    "weight_exponent_width": new_layer_exponent_width,
                    "weight_exponent_bias": new_layer_exponent_bias,
                    "weight_block_size": new_layer_block_size,
                    "bias_width": new_layer_data_width,
                    "bias_exponent_width": new_layer_exponent_width,
                    "bias_exponent_bias": new_layer_exponent_bias,
                    "bias_block_size": new_layer_block_size,
                }

            elif new_layer_cls == LinearBlockMinifloat:
                continue # broken

            elif new_layer_cls == LinearBlockLog:
                data_width = [8, 16, 32]
                shared_exponent_bias = [2, 4, 8]
                block_size = [[8], [16], [32]]

                new_layer_data_width = data_width[trial.suggest_int(f"{name}_data_width", 0, len(data_width)-1)]
                new_layer_exponent_bias = shared_exponent_bias[trial.suggest_int(f"{name}_shared_exponent_bias", 0, len(shared_exponent_bias)-1)]
                new_layer_block_size = block_size[trial.suggest_int(f"{name}_block_size", 0, len(block_size)-1)]

                kwargs["config"] = {
                    "data_in_width": new_layer_data_width,
                    "data_in_exponent_bias_width": new_layer_exponent_bias,
                    "data_in_block_size": new_layer_block_size,
                    "weight_width": new_layer_data_width,
                    "weight_exponent_bias_width": new_layer_exponent_bias,
                    "weight_block_size": new_layer_block_size,
                    "bias_width": new_layer_data_width,
                    "bias_exponent_bias_width": new_layer_exponent_bias,
                    "bias_block_size": new_layer_block_size,
                }

            elif new_layer_cls == LinearBinary:
                new_layer_stochastic = bool([trial.suggest_int(f"{name}_stochastic", 0, 1)])
                new_layer_bipolar = True

                kwargs["config"] = {
                    "weight_stochastic": new_layer_stochastic,
                    "weight_bipolar": new_layer_bipolar,
                }

            elif new_layer_cls == LinearBinaryScaling:
                new_layer_stochastic = bool([trial.suggest_int(f"{name}_stochastic", 0, 1)])
                new_layer_bipolar = True
                new_layer_binary_training = True

                kwargs["config"] = {
                    "data_in_stochastic": new_layer_stochastic,
                    "data_in_bipolar": new_layer_bipolar,
                    "weight_stochastic": new_layer_stochastic,
                    "weight_bipolar": new_layer_bipolar,
                    "bias_stochastic": new_layer_stochastic,
                    "bias_bipolar": new_layer_bipolar,
                    "binary_training": new_layer_binary_training,
                }

            elif new_layer_cls == LinearBinaryResidualSign:
                continue # not implemented

            # replace layer
            new_layer = new_layer_cls(**kwargs)
            new_layer.weight.data = layer.weight.data
            deepsetattr(trial_model, name, new_layer)

    return trial_model

In [ ]:
def objective_from_search_space(search_space):

  def objective(trial):
      # model, train, evaluate
      model = construct_model(trial, search_space)
      trainer = get_trainer(model=model, tokenized_dataset=dataset, tokenizer=tokenizer, evaluate_metric="accuracy", num_train_epochs=1)
      trainer.train()
      eval_results = trainer.evaluate()

      # save model
      trial.set_user_attr("model", model)

      return eval_results["eval_accuracy"]

  return objective

In [ ]:
from pathlib import Path
from google.colab import files
import pandas as pd

df = pd.DataFrame(columns=["trial", "acc", "precision"])

for k, v in precision_dict.items():
  # create search space and objective function
  search_space = { "linear_layer_choices": [torch.nn.Linear, v] }
  obj = objective_from_search_space(search_space)

  # create and run study
  sampler = TPESampler()
  study = optuna.create_study(direction="maximize", study_name=f"bert-tiny-nas-study-{k}", sampler=sampler)
  study.optimize(obj, n_trials=100, timeout=3600*24)

  # save all trial information
  cdf = study.trials_dataframe()
  cdf = cdf[["number", "value"]]
  cdf = cdf.rename(columns={"number": "trial", "value": "acc"})
  cdf["precision"] = k
  df = pd.concat([df, cdf])

df.to_pickle(f"{Path.home()}/tutorial6_task2.pickle")
files.download(f"{Path.home()}/tutorial6_task2.pickle")